OUR DATA:

In [10]:
import pandas as pd
df = pd.read_csv("data.csv", sep = ";")
df

,loop_iteration,cycle_iteration,money,bet,bets in cycle,all-in,all-in-money,lost_cycle
0,1,1,10000,1,1,False,0,False
1,1,2,10001,1,1,False,0,False
2,1,3,10002,1,1,False,0,False
3,1,4,10003,1,3,False,0,False
4,1,5,10004,1,4,False,0,False
...,...,...,...,...,...,...,...,...
6187682,100,2,10999,999,1,False,0,False
6187683,100,3,11998,999,1,False,0,False
6187684,100,4,12997,999,1,False,0,False
6187685,100,5,13996,999,3,False,0,False


lets try simplify this dataframe

In [3]:
data = []

for i in range(1,1000):
    for j in range(1,100):
        info = {
        "bet":i,
        "loop_iteration":j,
        "won":True
        }
        if True in df[(df["bet"]==i)&(df["loop_iteration"]==j)]["lost_cycle"].values:
            info["won"] = False
        data.append(info)
df2 = pd.DataFrame(data=data)
df2


KeyboardInterrupt



We cant do it like that becuase it took me around 28 minutes which is very suboptimal

In [5]:
df3 = df.groupby(by=["bet","loop_iteration"]).agg({
    "money": 'last',
    "all-in": 'sum',
    "cycle_iteration":'max',
    "lost_cycle":'sum',
    "bets in cycle": "mean"

}
)
df3.rename(columns={"lost_cycle":"lost_game", "bets in cycle":"avg_bets_in_cycle"},inplace=True)
df3['lost_game'].astype(bool)
df3

money  all-in  cycle_iteration  lost_game  \
bet loop_iteration                                              
1   1               19999       1            15077          0   
    2                2424       4             3308          1   
    3               19999       0            10000          0   
    4                 848       2             6869          1   
    5               19999       1            11533          0   
...                   ...     ...              ...        ...   
999 96              10999       1                2          1   
    97              19001       1               14          0   
    98              19990       0               11          0   
    99              11998       1                3          1   
    100             14995       1                6          1   

                    avg_bets_in_cycle  
bet loop_iteration                     
1   1                        2.018240  
    2                        2.051693  
    3                        2.019400  
    4                        2.011355  
    5                        1.973207  
...                               ...  
999 96                       2.500000  
    97                       1.642857  
    98                       1.818182  
    99                       2.333333  
    100                      2.000000  

[99900 rows x 5 columns]

great, we got the same or even better(becuase we have more room to add unique columns with ease) results.

Let's check winratio for each bet.


In [7]:
#winratio overall

df_winratio = df3.groupby(by="bet").agg({
    "lost_game": "mean"
})
df_winratio.rename(columns={"lost_game":"win_ratio"},inplace=True)

df_winratio

,win_ratio
bet,
1,0.48
2,0.49
3,0.51
4,0.52
5,0.55
...,...
995,0.49
996,0.56
997,0.55


Ok, now let's try something harder.
How my implemented all-in strategy influence the expected value of money.

To calculate that we need information how many games we manage to rescue, and how much money we could save if we wouldn't use all-in strategy.


In [8]:

all_in_save = df[(df["lost_cycle"]==False) & (df["all-in"]==True) ][['bet', 'loop_iteration']]
all_in_save

,bet,loop_iteration
1306,1,1
15443,1,2
15886,1,2
17121,1,2
35132,1,4
...,...,...
6187509,999,79
6187519,999,79
6187626,999,93
6187635,999,94


Dataframe with bet and loop iteration of saved cycles by all in. Now we need to figure out how to check if the game was saved

In [9]:
lost_games = df3[(df3["lost_game"]==True)]
lost_games

money  all-in  cycle_iteration  lost_game  \
bet loop_iteration                                              
1   2                2424       4             3308          1   
    4                 848       2             6869          1   
    6               13093       1             3094          1   
    7               14489       1             4490          1   
    8               15871       1             5872          1   
...                   ...     ...              ...        ...   
999 93              14006       2                9          1   
    94              11009       2                6          1   
    96              10999       1                2          1   
    99              11998       1                3          1   
    100             14995       1                6          1   

                    avg_bets_in_cycle  
bet loop_iteration                     
1   2                        2.051693  
    4                        2.011355  
    6                        1.991597  
    7                        1.991537  
    8                        1.967302  
...                               ...  
999 93                       2.000000  
    94                       2.166667  
    96                       2.500000  
    99                       2.333333  
    100                      2.000000  

[50482 rows x 5 columns]

finding common part by pd.merge


In [14]:
saved_by_all_in = pd.merge(all_in_save,lost_games, on=["bet","loop_iteration"],how="left", indicator=True)

saved_by_all_in

,bet,loop_iteration,money,all-in,cycle_iteration,lost_game,avg_bets_in_cycle,_merge
0,1,1,NaN,NaN,NaN,NaN,NaN,left_only
1,1,2,2424.0,4.0,3308.0,1.0,2.051693,both
2,1,2,2424.0,4.0,3308.0,1.0,2.051693,both
3,1,2,2424.0,4.0,3308.0,1.0,2.051693,both
4,1,4,848.0,2.0,6869.0,1.0,2.011355,both
...,...,...,...,...,...,...,...,...
50100,999,79,40.0,2.0,12.0,1.0,2.250000,both
50101,999,79,40.0,2.0,12.0,1.0,2.250000,both
50102,999,93,14006.0,2.0,9.0,1.0,2.000000,both
50103,999,94,11009.0,2.0,6.0,1.0,2.166667,both


this apear like very illegible, but we want only bet and loop iteration of situation where all in saved the game _merge=left_only(for calculating win ratio, and the saved money by using this strategy), and where all_in didnt save te game _merge=both (for calculating lost money by using this strategy).

There will be much more both becuase there could be a lot of all in plays in lost game.
